# 02 数据清洗与存储管理

本 Notebook 完成单表清洗、宽表/长表转换、多表合并，并额外保存 Parquet 格式用于和 CSV 对比。


In [1]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
for directory in ["data/clean", "data/combined", "output"]:
    os.makedirs(ROOT / directory, exist_ok=True)

stock_info = pd.read_csv(ROOT / "data/stock_list.csv")
stock_info


,code,name,industry,reason
0,1,平安银行,银行,全国性股份制银行，金融板块代表性强。
1,600036,招商银行,银行,零售银行龙头，盈利质量较高。
2,600519,贵州茅台,白酒,白酒行业龙头，消费属性突出。
3,858,五粮液,白酒,高端白酒代表，适合行业内比较。
4,2594,比亚迪,汽车,新能源汽车龙头，成长性强。
5,601633,长城汽车,汽车,自主品牌车企，处于新能源转型阶段。
6,601857,中国石油,能源,能源央企代表，受油价和周期影响。
7,600900,长江电力,能源,现金流稳定，防御属性较强。
8,600941,中国移动,通讯,通讯运营商龙头，稳健和高股息特征明显。
9,63,中兴通讯,通讯,通讯设备代表，技术周期敏感。


## 3.1 单表清洗

对每只股票分别检查缺失值、日期格式、数值类型、重复记录，并标注单日收益率超过 ±20% 的极端波动。


In [2]:
def missing_profile(df: pd.DataFrame, code: str) -> pd.DataFrame:
    return pd.DataFrame({
        "code": code,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(4),
    })


def clean_stock_file(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    code = path.stem.replace("stock_", "")
    raw = pd.read_csv(path)
    before_rows = len(raw)
    missing = missing_profile(raw, code)

    df = raw.copy()
    df["date"] = pd.to_datetime(df["date"])
    numeric_cols = ["open", "close", "high", "low", "volume", "amount"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    duplicate_count = df.duplicated(subset=["date", "code"]).sum()
    df = df.drop_duplicates(subset=["date", "code"], keep="last").sort_values("date")
    df[numeric_cols] = df[numeric_cols].ffill()
    df["return"] = np.log(df["close"] / df["close"].shift(1))
    df["is_extreme"] = df["return"].abs() > 0.20
    df = df.merge(stock_info[["code", "industry"]], on="code", how="left")
    audit = {
        "code": code,
        "before_rows": before_rows,
        "after_rows": len(df),
        "duplicates_removed": int(duplicate_count),
        "extreme_rows": int(df["is_extreme"].sum()),
    }
    return df, missing, audit


clean_frames = []
missing_frames = []
audit_records = []

for path in sorted((ROOT / "data/stock").glob("stock_*.csv")):
    cleaned, missing, audit = clean_stock_file(path)
    clean_frames.append(cleaned)
    missing_frames.append(missing)
    audit_records.append(audit)

stock_clean = pd.concat(clean_frames, ignore_index=True)
missing_summary = pd.concat(missing_frames, ignore_index=True)
clean_audit = pd.DataFrame(audit_records)

stock_clean.to_csv(ROOT / "data/clean/stock_clean.csv", index=False, encoding="utf-8-sig")
missing_summary.to_csv(ROOT / "data/clean/missing_summary.csv", index=False, encoding="utf-8-sig")
clean_audit.to_csv(ROOT / "data/clean/clean_audit.csv", index=False, encoding="utf-8-sig")

clean_audit


,code,before_rows,after_rows,duplicates_removed,extreme_rows
0,000001,1541,1541,0,0
1,000063,1539,1539,0,0
2,000858,1541,1541,0,0
3,002594,1540,1540,0,0
4,600036,1541,1541,0,0
5,600519,1541,1541,0,0
6,600900,1529,1529,0,0
7,600941,1053,1053,0,0
8,601633,1540,1540,0,0
9,601857,1540,1540,0,0


清洗说明：缺失值先统计再用向前填充处理，原因是日度行情中的少量缺失通常来自停牌、节假日或数据源临时缺口，向前填充可以保持时间序列连续性。重复记录按 `code + date` 删除，只保留最后一条。极端收益率不删除，只用 `is_extreme` 标注，避免把真实涨跌停或复权异常直接抹掉。


## 3.2 宽表与长表转换

宽表适合相关系数、矩阵运算和多资产组合计算；长表适合分组统计、绘图分面和与行业信息合并。


In [3]:
close_wide = stock_clean.pivot_table(index="date", columns="code", values="close", aggfunc="last").sort_index()
close_long = close_wide.reset_index().melt(id_vars="date", var_name="code", value_name="close")

close_wide.to_csv(ROOT / "data/clean/stock_close_wide.csv", encoding="utf-8-sig")
close_long.to_csv(ROOT / "data/clean/stock_close_long.csv", index=False, encoding="utf-8-sig")

print("Wide table shape:", close_wide.shape)
print("Long table shape:", close_long.shape)
close_long.head()


Wide table shape: (1541, 10)
Long table shape: (15410, 3)


,date,code,close
0,2020-01-02,1,2189.30
1,2020-01-03,1,2228.06
2,2020-01-06,1,2214.31
3,2020-01-07,1,2224.31
4,2020-01-08,1,2163.05


## 3.3 多表合并

股票日度数据先与指数按日期左连接，再将月度宏观指标映射到同一月份的每个交易日。


In [4]:
def load_index(code: str, prefix: str) -> pd.DataFrame:
    df = pd.read_csv(ROOT / f"data/index/index_{code}.csv")
    df["date"] = pd.to_datetime(df["date"])
    keep = df[["date", "close"]].rename(columns={"close": f"{prefix}_close"})
    keep[f"{prefix}_return"] = np.log(keep[f"{prefix}_close"] / keep[f"{prefix}_close"].shift(1))
    return keep


hs300 = load_index("000300", "hs300")
zz500 = load_index("000905", "zz500")

before_stock_rows = len(stock_clean)
combined = stock_clean.merge(hs300, on="date", how="left")
after_hs300_rows = len(combined)
combined = combined.merge(zz500, on="date", how="left")
after_zz500_rows = len(combined)

macro_frames = []
for path in sorted((ROOT / "data/macro").glob("macro_*.csv")):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    macro_frames.append(df)
macro = pd.concat(macro_frames, ignore_index=True)
macro["month"] = macro["date"].dt.to_period("M").astype(str)
macro_month = (
    macro.sort_values("date")
    .drop_duplicates(subset=["month", "indicator"], keep="last")
    .pivot(index="month", columns="indicator", values="value")
    .reset_index()
)

combined["month"] = combined["date"].dt.to_period("M").astype(str)
before_macro_rows = len(combined)
combined = combined.merge(macro_month, on="month", how="left")
after_macro_rows = len(combined)

combined.to_csv(ROOT / "data/combined/combined_data.csv", index=False, encoding="utf-8-sig")

merge_audit = pd.DataFrame([
    {"step": "stock_clean", "rows": before_stock_rows, "note": "清洗后的股票长表"},
    {"step": "left_join_hs300", "rows": after_hs300_rows, "note": "按日期左连接沪深300，行数应保持不变"},
    {"step": "left_join_zz500", "rows": after_zz500_rows, "note": "按日期左连接中证500，行数应保持不变"},
    {"step": "left_join_macro", "rows": after_macro_rows, "note": "按月份映射宏观指标，行数应保持不变"},
])
merge_audit.to_csv(ROOT / "data/clean/merge_audit.csv", index=False, encoding="utf-8-sig")
merge_audit


,step,rows,note
0,stock_clean,14905,清洗后的股票长表
1,left_join_hs300,14905,按日期左连接沪深300，行数应保持不变
2,left_join_zz500,14905,按日期左连接中证500，行数应保持不变
3,left_join_macro,14905,按月份映射宏观指标，行数应保持不变


合并说明：所有连接均以个股日度记录为主表，所以行数理论上不应减少。若指数或宏观指标缺失，只会在对应字段留下空值，便于后续判断数据覆盖情况。


## Parquet 存储与 CSV 对比

Parquet 保留字段类型，支持只读取需要的列。下面对比 CSV 和 Parquet 的文件大小与读取时间。


In [5]:
csv_path = ROOT / "data/clean/stock_clean.csv"
parquet_path = ROOT / "data/clean/stock_clean.parquet"

stock_clean.to_parquet(parquet_path, index=False)

subset = pd.read_parquet(parquet_path, columns=["date", "code", "close"])
schema = pq.read_schema(parquet_path)
print(schema)
print("Column subset shape:", subset.shape)

t0 = time.time()
pd.read_csv(csv_path)
csv_time = time.time() - t0

t0 = time.time()
pd.read_parquet(parquet_path)
parquet_time = time.time() - t0

storage_compare = pd.DataFrame([
    {"format": "CSV", "read_seconds": csv_time, "size_kb": os.path.getsize(csv_path) / 1024},
    {"format": "Parquet", "read_seconds": parquet_time, "size_kb": os.path.getsize(parquet_path) / 1024},
])
storage_compare.to_csv(ROOT / "data/clean/storage_compare.csv", index=False, encoding="utf-8-sig")
storage_compare


date: timestamp[us]
code: int64
name: large_string
open: double
close: double
high: double
low: double
volume: double
amount: double
source: large_string
return: double
is_extreme: bool
industry: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1529
Column subset shape: (14905, 3)


,format,read_seconds,size_kb
0,CSV,0.053021,1830.994141
1,Parquet,0.003893,712.429688


对比结论：在本作业这种中小规模数据下，CSV 与 Parquet 的速度差异通常不会非常夸张；但 Parquet 的类型信息、压缩效果和列式读取能力更适合字段更多、数据量更大、只需读取部分列的分析场景。
